# Movie Recommendation System
Build a content-based recommender using movie information, cast and director features.

## 1. Import Libraries

In [3]:
#Import libraries
import pandas as pd
import ast
import pickle

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load and Check the Data

In [5]:
# Load the datasets
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("2tmdb_5000_credits.csv")

# Check dataset dimensions
print("Movies:", movies.shape)
print("Credits:", credits.shape)

Movies: (4803, 20)
Credits: (4803, 4)


In [6]:
# Inspect column names
print("Movies columns:", movies.columns.tolist())
print("Credits columns:", credits.columns.tolist())

Movies columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']
Credits columns: ['movie_id', 'title', 'cast', 'crew']


In [7]:
# Check missing and duplicate movie IDs
print("Missing movie IDs:", movies["id"].isna().sum())
print("Missing credits IDs:", credits["movie_id"].isna().sum())

print("Duplicate movie IDs:", movies["id"].duplicated().sum())
print("Duplicate credits IDs:", credits["movie_id"].duplicated().sum())

Missing movie IDs: 0
Missing credits IDs: 0
Duplicate movie IDs: 0
Duplicate credits IDs: 0


In [8]:
# Check movie ID matches between datasets
print("Movies without credits:",
    (~movies["id"].isin(credits["movie_id"])).sum()
)

print("Credits without movies:",
    (~credits["movie_id"].isin(movies["id"])).sum()
)

Movies without credits: 0
Credits without movies: 0


## 3. Merge the Datasets
Combine movie information and credits using unique movie IDs.

In [10]:
# Merge movie information with credits
merged_movies = movies.merge(credits, 
                             left_on="id", 
                             right_on="movie_id", 
                             how="inner", 
                             validate="one_to_one",
                             suffixes=("", "_credits")
)

print("Merged shape:", merged_movies.shape)

Merged shape: (4803, 24)


In [11]:
# Check title differences after merging
title_mismatch = merged_movies["title"] != merged_movies["title_credits"]

print("Different titles:", title_mismatch.sum())

Different titles: 0


## 4. Prepare the Data
Select the columns needed for content-based recommendations.

In [13]:
# Select recommendation features
selected_columns = [
    "id", "title", "overview", "genres", "keywords", "cast", "crew"
]

movie_features = merged_movies[selected_columns].copy()

movie_features.head(2)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [14]:
# Count missing values in the selected columns
movie_features.isna().sum()

id          0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [15]:
# Keep movies with missing overviews
movie_features["overview"] = movie_features["overview"].fillna("")

# Verify missing overviews
print(movie_features["overview"].isna().sum())

0


## 5. Prepare Text Features
Extract names from structured movie information.

In [17]:
# Inspect the first movie's genres
genre_list = ast.literal_eval(movie_features["genres"].iloc[0])

print(genre_list)

[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 878, 'name': 'Science Fiction'}]


In [18]:
# Extract genre names from one movie
genre_names = []

for genre in genre_list:
    genre_names.append(genre["name"])

print(genre_names)

['Action', 'Adventure', 'Fantasy', 'Science Fiction']


In [19]:
# Define a function to extract names
def extract_names(text):
    items = ast.literal_eval(text)
    names = []

    for item in items:
        names.append(item["name"])

    return names

In [20]:
extract_names(movie_features["genres"].iloc[0])

['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [21]:
# Extract genre names for all movies
movie_features["genres"] = movie_features["genres"].apply(extract_names)

movie_features[["title", "genres"]].head(2)

,title,genres
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]"
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]"


In [22]:
# Extract keyword names for all movies
movie_features["keywords"] = movie_features["keywords"].apply(extract_names)

movie_features[["title", "keywords"]].head(2)

,title,keywords
0,Avatar,"[culture clash, future, space war, space colon..."
1,Pirates of the Caribbean: At World's End,"[ocean, drug abuse, exotic island, east india ..."


In [23]:
# Inspect the first three cast records
cast_list = ast.literal_eval(movie_features["cast"].iloc[0])

print(cast_list[:3])

[{'cast_id': 242, 'character': 'Jake Sully', 'credit_id': '5602a8a7c3a3685532001c9a', 'gender': 2, 'id': 65731, 'name': 'Sam Worthington', 'order': 0}, {'cast_id': 3, 'character': 'Neytiri', 'credit_id': '52fe48009251416c750ac9cb', 'gender': 1, 'id': 8691, 'name': 'Zoe Saldana', 'order': 1}, {'cast_id': 25, 'character': 'Dr. Grace Augustine', 'credit_id': '52fe48009251416c750aca39', 'gender': 1, 'id': 10205, 'name': 'Sigourney Weaver', 'order': 2}]


In [24]:
# Extract the first three actor names from one movie
actor_names = []

for actor in cast_list[:3]:
    actor_names.append(actor["name"])

print(actor_names)

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']


In [25]:
# Define a function to extract the first three actor names
def extract_cast(text):
    actors = ast.literal_eval(text)
    names = []

    for actor in actors[:3]:
        names.append(actor["name"])

    return names

In [26]:
extract_cast(movie_features["cast"].iloc[0])

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']

In [27]:
# Extract the first three actor names for all movies
movie_features["cast"] = movie_features["cast"].apply(extract_cast)

movie_features[["title", "cast"]].head(2)

,title,cast
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]"


In [28]:
# Inspect one crew record from the first movie
crew_list = ast.literal_eval(movie_features["crew"].iloc[0])

print(crew_list[0])

{'credit_id': '52fe48009251416c750aca23', 'department': 'Editing', 'gender': 0, 'id': 1721, 'job': 'Editor', 'name': 'Stephen E. Rivkin'}


In [29]:
# Find the director of the first movie
for person in crew_list:
    if person["job"] == "Director":
        print(person["name"])

James Cameron


In [30]:
# Define a function to extract director names
def extract_director(text):
    crew = ast.literal_eval(text)
    names = []

    for person in crew:
        if person["job"] == "Director":
            names.append(person["name"])

    return names

In [31]:
extract_director(movie_features["crew"].iloc[0])

['James Cameron']

In [32]:
# Extract director names for all movies
movie_features["director"] = movie_features["crew"].apply(extract_director)

movie_features[["title", "director"]].head(2)

,title,director
0,Avatar,[James Cameron]
1,Pirates of the Caribbean: At World's End,[Gore Verbinski]


In [33]:
# Split movie overviews into word lists
movie_features["overview"] = movie_features["overview"].str.split()

movie_features["overview"].iloc[0]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.']

## 6. Create Movie Tags
Combine overview, genres, keywords, cast and director into one list per movie.

In [35]:
# Combine movie content features
movie_features["tags"] = (
    movie_features["overview"]
    + movie_features["genres"]
    + movie_features["keywords"]
    + movie_features["cast"]
    + movie_features["director"]
)

movie_features["tags"].iloc[0]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.',
 'Action',
 'Adventure',
 'Fantasy',
 'Science Fiction',
 'culture clash',
 'future',
 'space war',
 'space colony',
 'society',
 'space travel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alien planet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'love affair',
 'anti war',
 'power relations',
 'mind and soul',
 '3d',
 'Sam Worthington',
 'Zoe Saldana',
 'Sigourney Weaver',
 'James Cameron']

In [36]:
# Keep multiword tag items together
cleaned_tags = []

for tag in movie_features["tags"].iloc[0]:
    cleaned_tags.append(tag.replace(" ", ""))

print(cleaned_tags[-4:])

['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver', 'JamesCameron']


In [37]:
# Define a function to remove spaces within tag items
def remove_tag_spaces(tags):
    cleaned_tags = []

    for tag in tags:
        cleaned_tags.append(tag.replace(" ", ""))

    return cleaned_tags

In [38]:
remove_tag_spaces(movie_features["tags"].iloc[0])[-4:]

['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver', 'JamesCameron']

In [39]:
# Remove spaces within tag items for all movies
movie_features["tags"] = movie_features["tags"].apply(remove_tag_spaces)

print(movie_features["tags"].iloc[0][-4:])

['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver', 'JamesCameron']


In [40]:
# Join tag items into one text per movie
movie_features["tags"] = movie_features["tags"].str.join(" ")

print(movie_features["tags"].iloc[0])

In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron


## 7. Vectorize Movie Tags
Convert prepared movie tags into word count vectors.

In [42]:
# Convert movie tags into word count vectors
vectorizer = CountVectorizer(max_features=500, stop_words="english")
vectors = vectorizer.fit_transform(movie_features["tags"])

print(vectors.shape)

(4803, 500)


In [43]:
# Inspect words in the vocabulary
feature_names = vectorizer.get_feature_names_out()

print(feature_names[:10])

['1970s' '3d' 'accident' 'action' 'adventure' 'affair'
 'aftercreditsstinger' 'age' 'agent' 'airplane']


## 8. Calculate Movie Similarity
Compare movie vectors using cosine similarity.

In [45]:
# Calculate similarity between all movie vectors
similarity = cosine_similarity(vectors)

print(similarity.shape)

(4803, 4803)


In [46]:
# Inspect similarity scores for Avatar
print("Avatar with itself:", similarity[0, 0])
print("Avatar with the second movie:", similarity[0, 1])

Avatar with itself: 1.0
Avatar with the second movie: 0.2193817272381392


In [47]:
# Pair movie positions with similarity scores for Avatar
movie_scores = list(enumerate(similarity[0]))

print(movie_scores[:5])

[(0, 1.0), (1, 0.2193817272381392), (2, 0.12524485821702988), (3, 0.04756514941544941), (4, 0.32539568672798425)]


In [48]:
# Rank movies by similarity score
sorted_scores = sorted(
    movie_scores,
    key=lambda item: item[1],
    reverse=True
)

print(sorted_scores[:6])

[(0, 1.0), (507, 0.568796458994521), (151, 0.4950737714883372), (322, 0.4900980294098034), (1191, 0.4900980294098034), (1213, 0.48507125007266594)]


In [49]:
# Display the title of the highest-ranked other movie
movie_position = sorted_scores[1][0]

print(movie_features.iloc[movie_position]["title"])

Independence Day


In [50]:
# Exclude Avatar and select five recommendations
other_movies = []

for movie_position, score in sorted_scores:
    if movie_position != 0:
        other_movies.append((movie_position, score))

top_five = other_movies[:5]

# Display recommended movie titles
for movie_position, score in top_five:
    title = movie_features.iloc[movie_position]["title"]
    print(title, round(score, 3))

Independence Day 0.569
Beowulf 0.495
The Fifth Element 0.49
Small Soldiers 0.49
Aliens vs Predator: Requiem 0.485


In [51]:
# Find the movie selected by its title
selected_title = "Pirates of the Caribbean: At World's End"
title_matches = movie_features["title"] == selected_title

print(title_matches.head())

0    False
1     True
2    False
3    False
4    False
Name: title, dtype: bool


In [52]:
# Find the selected movie's row position
movie_positions = [
    position
    for position, matches in enumerate(title_matches)
    if matches
]

print(movie_positions)

[1]


In [53]:
# Get similarity scores for the selected movie
selected_position = movie_positions[0]
selected_scores = similarity[selected_position]

print(selected_scores[:5])

[0.21938173 1.         0.15569979 0.05913124 0.20225996]


In [54]:
# Pair movie positions with the selected movie's scores
movie_scores = list(enumerate(selected_scores))

print(movie_scores[:5])

[(0, 0.2193817272381392), (1, 1.0000000000000002), (2, 0.1556997888323046), (3, 0.059131239598908265), (4, 0.20225995873897262)]


In [55]:
# Rank movies by similarity score
sorted_scores = sorted(
    movie_scores,
    key=lambda item: item[1],
    reverse=True
)

print(sorted_scores[:6])

[(1, 1.0000000000000002), (12, 0.5118906968889915), (98, 0.4545454545454546), (17, 0.42379344841321004), (3568, 0.4029114820126901), (232, 0.40201512610368484)]


In [56]:
# Exclude the selected movie from recommendations
other_movies = []

for movie_position, score in sorted_scores:
    if movie_position != selected_position:
        other_movies.append((movie_position, score))

top_five = other_movies[:5]

print(top_five)

[(12, 0.5118906968889915), (98, 0.4545454545454546), (17, 0.42379344841321004), (3568, 0.4029114820126901), (232, 0.40201512610368484)]


In [57]:
# Display recommended movie titles and scores
for movie_position, score in top_five:
    title = movie_features.iloc[movie_position]["title"]
    print(title, round(score, 3))

Pirates of the Caribbean: Dead Man's Chest 0.512
The Hobbit: An Unexpected Journey 0.455
Pirates of the Caribbean: On Stranger Tides 0.424
20,000 Leagues Under the Sea 0.403
The Wolverine 0.402


In [58]:
# Define a function to find movie positions by title
def recommend_movies(movie_title):
    title_matches = movie_features["title"] == movie_title
    movie_positions = []

    for position, matches in enumerate(title_matches):
        if matches:
            movie_positions.append(position)
    # Check whether the movie title exists
    if not movie_positions:
        raise ValueError("Movie title not found.")
        # Reject ambiguous movie titles
    if len(movie_positions) > 1:
        raise ValueError("Multiple movies share this title. Select by movie ID.")
    
    # Get similarity scores for the selected movie
    selected_position = movie_positions[0]
    selected_scores = similarity[selected_position]

        # Pair movie positions with similarity scores
    movie_scores = list(enumerate(selected_scores))

    # Rank movies by similarity score
    sorted_scores = sorted(
        movie_scores,
        key=lambda item: item[1],
        reverse=True
    )

        # Exclude the selected movie
    other_movies = []

    for movie_position, score in sorted_scores:
        if movie_position != selected_position:
            other_movies.append((movie_position, score))

       # Get titles and scores for the five recommendations
    recommendations = []

    for movie_position, score in other_movies[:5]:
        title = movie_features.iloc[movie_position]["title"]
        recommendations.append((title, float(score)))

    return recommendations

In [59]:
# Test recommendation titles and scores
recommend_movies("Avatar")

[('Independence Day', 0.568796458994521),
 ('Beowulf', 0.4950737714883372),
 ('The Fifth Element', 0.4900980294098034),
 ('Small Soldiers', 0.4900980294098034),
 ('Aliens vs Predator: Requiem', 0.48507125007266594)]

In [60]:
# Test recommendations for another movie
recommend_movies("Pirates of the Caribbean: At World's End")

[("Pirates of the Caribbean: Dead Man's Chest", 0.5118906968889915),
 ('The Hobbit: An Unexpected Journey', 0.4545454545454546),
 ('Pirates of the Caribbean: On Stranger Tides', 0.42379344841321004),
 ('20,000 Leagues Under the Sea', 0.4029114820126901),
 ('The Wolverine', 0.40201512610368484)]

## 10. Save Recommendation Data
Prepare and save the movie table and similarity matrix for the Flask app.

In [62]:
# Prepare movie data for saving
movies_to_save = movie_features[["id", "title", "tags"]].copy()

# Check dimensions before saving
print("Movie data:", movies_to_save.shape)
print("Similarity matrix:", similarity.shape)

Movie data: (4803, 3)
Similarity matrix: (4803, 4803)


In [63]:
# Save the prepared movie table
with open("model.pkl", "wb") as file:
    pickle.dump(movies_to_save, file)

# Save the similarity matrix
with open("similarity.pkl", "wb") as file:
    pickle.dump(similarity, file)

print("Files saved successfully.")

Files saved successfully.


In [64]:
# Reload the saved movie table
with open("model.pkl", "rb") as file:
    loaded_movies = pickle.load(file)

# Reload the saved similarity matrix
with open("similarity.pkl", "rb") as file:
    loaded_similarity = pickle.load(file)

print("Loaded movies:", loaded_movies.shape)
print("Loaded similarity:", loaded_similarity.shape)

Loaded movies: (4803, 3)
Loaded similarity: (4803, 4803)


In [65]:
# Check whether Flask is installed
from importlib.metadata import version

print("Flask version:", version("flask"))

Flask version: 3.0.3


In [66]:
# Back up the starter Flask app
from pathlib import Path

project_folder = Path.cwd().parent
app_path = project_folder / "app.py"
backup_path = project_folder / "app_starter_backup.txt"

if not backup_path.exists():
    starter_code = app_path.read_text(encoding="utf-8")
    backup_path.write_text(starter_code, encoding="utf-8")

print("Starter app backup is ready.")

Starter app backup is ready.


In [67]:
# Check installed package versions
from importlib.metadata import version

packages = ["Flask", "pandas", "numpy", "scikit-learn"]

for package in packages:
    print(package, version(package))

Flask 3.0.3
pandas 2.2.2
numpy 1.26.4
scikit-learn 1.5.1


In [127]:
# Check saved file sizes
from pathlib import Path

for filename in ["model.pkl", "similarity.pkl"]:
    size_mb = Path(filename).stat().st_size / (1024 * 1024)
    print(filename, round(size_mb, 2), "MB")

model.pkl 2.25 MB
similarity.pkl 176.0 MB


In [129]:
# Save a compressed copy of the similarity matrix
import gzip

with gzip.open("similarity.pkl.gz", "wb") as file:
    pickle.dump(similarity, file)

size_mb = Path("similarity.pkl.gz").stat().st_size / (1024 * 1024)
print("Compressed similarity:", round(size_mb, 2), "MB")

Compressed similarity: 31.57 MB
